# Lesson 12 — Scaling up safely

**Moodle section:** 12. Scaling up: larger CSV datasets

Use the course cycle: **Predict → Run → Check → Change → Explain**. Save before closing Python Lab; your notebook remains in your server workspace.


## Goals

Generate deterministic data, select schema, process chunks, merge group totals, and reconcile row counts.

> The generated CSV is saved in your persistent **work** area. The generator is located safely even when this notebook starts from a different folder.


In [ ]:
from pathlib import Path
import subprocess
import sys


def find_course_data(filename):
    """Find a course data file without depending on the notebook's start folder."""
    roots = [
        Path.cwd(),
        *Path.cwd().parents,
        Path.home() / "work",
        Path("/opt/python-lab/course-materials"),
    ]
    candidates = []
    for root in roots:
        candidates.extend((root / "data" / filename, root / filename))

    checked = []
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.is_file():
            return candidate

    locations = "\n".join(f"- {candidate}" for candidate in checked)
    raise FileNotFoundError(
        f"Course data file {filename!r} was not found. Checked:\n{locations}"
    )


generator = find_course_data("generate-learning-centre-data.py")
generated_dir = Path.home() / "work" / "python-course-generated-data"
generated_dir.mkdir(parents=True, exist_ok=True)
large_file = generated_dir / "learning-centres-10000.csv"
if not large_file.exists():
    subprocess.run([
        sys.executable, str(generator),
        "--rows", "10000", "--output", str(large_file)
    ], check=True)
print("Generator:", generator.resolve())
print("Data file:", large_file.resolve())
large_file


In [ ]:
import pandas as pd

totals = {}
row_count = 0
for chunk in pd.read_csv(
    large_file,
    usecols=["district", "material_cost"],
    dtype={"district": "string", "material_cost": "float64"},
    chunksize=2_000,
):
    row_count += len(chunk)
    part = chunk.groupby("district")["material_cost"].sum()
    for district, amount in part.items():
        totals[district] = totals.get(district, 0) + amount

print("Rows processed:", row_count)
print(totals)

## Transfer challenge

Process registered and attended totals by district in chunks. Reconcile the processed row count, then calculate district attendance rates.

In [ ]:
# Test with 10,000 rows before increasing the generated size.


**Check:** Never average unweighted chunk means. Merge sums and counts, then calculate the final statistic.

## Learning record

Before returning to Moodle, write down: (1) one result you predicted correctly, (2) one change you tested, and (3) one question or error you still have. Then complete the Moodle learning check.
